# Cat state under the ATS pump: chaotic vs stabilized regime

A two-photon-pumped ATS (memory `a`, buffer `b`), simulated in the lab frame
with `dq.mesolve`. The memory starts in a cat, the buffer in vacuum. The
final memory state is shown as a Wigner function in the frame rotating at
`(w_d + w_p) / 2`.

This notebook used to run on `chuc`, on a local GPU with
`CUDA_VISIBLE_DEVICES` pinned by hand and a `Plot_setting` module from a home
directory. It now runs on the anb-compute Ray cluster. Each simulation is one
picklable function that runs on a GPU worker and returns plain numpy (the
reduced states, the Wigner grid, the scalars). Only the plotting happens here.
The original is kept in `legacy_chuc/`.

## Running it

1. **Kernel**: the workspace pixi environment,
   `/workspace/.pixi/envs/default/bin/python` (**Python 3.13.x**, `ray 2.56.0`).
   These must match the cluster.
2. **Vault session**, once (lasts about a week):

   ```bash
   export VAULT_ADDR=https://vault.int.alice-bob.com
   vault login -method=oidc role=research_core_user
   ```

3. Run all cells. The first `acr.run` of a session may open an interactive
   Microsoft device-code login for GCS read access. That is expected, and the
   login is cached.

If you have already run something that imports `ray` in this kernel, restart it
first. Ray caches `RAY_AUTH_MODE` at first import.

Nothing needs shipping: the GPU image's `prod-gpu` environment already has
jax (CUDA), dynamiqs 0.3.6 and scipy, and the functions below travel by value.
dynamiqs 0.3.6 takes solver options as keyword arguments, so
`mesolve(..., options=dq.Options(save_extra=..., save_states=False))` became
`mesolve(..., save_extra=..., save_states=False)`.


In [ ]:
import os
import subprocess
import sys

assert sys.version_info[:2] == (3, 13), (
    f"kernel runs Python {sys.version.split()[0]}; the cluster needs 3.13.x. "
    "Select the workspace pixi environment as the kernel."
)

# Ray reads RAY_AUTH_MODE once, at its first import, and caches it. Setting it
# here works *only* because nothing has imported ray yet in a fresh kernel.
os.environ["RAY_AUTH_MODE"] = "token"
os.environ.setdefault("VAULT_ADDR", "https://vault.int.alice-bob.com")

if not os.environ.get("RAY_AUTH_TOKEN"):
    try:
        os.environ["RAY_AUTH_TOKEN"] = subprocess.run(
            ["vault", "kv", "get", "-field=token", "kv-it/prod/ray/auth-token"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError) as exc:
        raise RuntimeError(
            "Could not mint the cluster token from Vault. In a terminal, run:\n"
            "  export VAULT_ADDR=https://vault.int.alice-bob.com\n"
            "  vault login -method=oidc role=research_core_user\n"
            "then re-run this cell (no kernel restart needed).\n"
            f"{(getattr(exc, 'stderr', '') or '').strip()}"
        ) from exc

import ray

from anb_compute import ray as acr

print("python", sys.version.split()[0], "| ray", ray.__version__)

In [ ]:
import numpy as np

# --- constants (computed once) ---
ħ = 1
e = 1
phi_quantum = 1/2

w_a = 25.338776456203686 #GHz
w_b_0 = 2*w_a#7.05 * 2 * np.pi #GHz
w_b = w_b_0 #GHz

phi_a = 0.11
Z_a = 2 * phi_quantum**2 * phi_a**2
C_a_inv = Z_a * w_a
E_Ca = C_a_inv / 2

phi_b = 0.204
Z_b = 2 * phi_quantum**2 * phi_b**2
C_b_inv = Z_b * w_b
E_Cb = C_b_inv / 2

E_J = 37.12 * 2 * np.pi *1.5
kappa = 2 * np.pi/10.4

# dynamiqs' default, i.e. what the original ran with (its outputs are float32).
# "double" for complex128 throughout.
PRECISION = "single"


### One simulation, on one GPU worker

`run_cat` is the original `run_simulation`, taking `epsilon_0` as an argument
instead of reading it, `epsilon_d` and `tsave` from globals. `epsilon_d =
ratio * g2`, with `ratio = 10` as in the original. jax and dynamiqs are imported
inside, on the worker.

With `w_b = 2 w_a` and the drive at `w_d = w_b`, the pump is static (`w_p = 2
w_a - w_d = 0`), so `g2 = sin(epsilon_0) E_J phi_a^2 phi_b`. `J_1(epsilon_0)`
in place of `sin(epsilon_0)` holds only for `w_p != 0`. It was used here until
2026-10-12, and underestimated `g2` by about 2x (`J_1 / sin = 0.50` at
`epsilon_0 = 0.12`, `0.57` at `pi/2`).

The memory starts in the normalized cat `|i alpha> + |-i alpha>`, with
`alpha^2 = epsilon_d / (2 g2) = ratio / 2` (`alpha = sqrt(5)` at `ratio = 10`):
the drive is `epsilon_d / 2` after the RWA. The red circle on the Wigner plots
has the same radius. The two are returned as `alpha_init` and `alpha_circle`.
The original started from `alpha = 1j * epsilon_d / (4 g2)` (`2.5j`), which is
not the cat size.

`check_truncation` is the original `run_single`: the commutator check
`max_t |1 - <[a, a^dag]>|`, from Fock state `|1, 0>`. The original notebook
defined it but never called it. It is run at the end, only if
`RUN_TRUNCATION_CHECK` is set.

In [ ]:
def run_cat(*, label, epsilon_0, n_a, T, n_time, ratio=10.0, precision="single"):
    """`run_simulation` for one `epsilon_0`, on one GPU worker. Returns plain numpy."""
    import time

    import jax
    import jax.numpy as jnp
    import numpy as np

    import dynamiqs as dq

    t_start = time.time()
    dq.set_precision(precision)

    # Static pump (w_p = 2 w_a - w_d = 0): sin(epsilon_0), not J_1(epsilon_0).
    g2 = np.sin(epsilon_0) * E_J * phi_a**2 * phi_b
    epsilon_d = ratio * g2
    kappa_2 = 4 * g2**2 / kappa
    tsave = jnp.linspace(0, T, n_time)

    n_b = (n_a + 1) // 2
    a = dq.destroy(n_a)
    b = dq.destroy(n_b)

    varphi_a_op_tot = dq.tensor(jnp.sqrt(Z_a/2)*(a + a.dag()) / phi_quantum, dq.eye(n_b))
    varphi_b_op_tot = dq.tensor(dq.eye(n_a), jnp.sqrt(Z_b/2)*(b + b.dag()) / phi_quantum)
    # alpha^2 = (epsilon_d / 2) / g2: the drive is epsilon_d / 2 after the RWA.
    alpha = 1j * np.sqrt(epsilon_d / (2 * g2))
    cat_state = dq.unit(dq.coherent(n_a, alpha) + dq.coherent(n_a, -alpha))
    psi_0 = dq.tensor(cat_state, dq.fock(n_b, 0))

    w_d = w_b_0
    drive = lambda t: epsilon_d * jnp.cos(w_d * t)
    H_drive = dq.modulated(drive, dq.tensor(dq.eye(n_a), b + b.dag()))

    w_p=2*w_a-w_d
    pump = lambda t: - 2 * E_J * jnp.sin(epsilon_0 * jnp.cos(w_p * t))
    H_pump = dq.modulated(pump, dq.sinm(varphi_a_op_tot + varphi_b_op_tot) - varphi_a_op_tot - varphi_b_op_tot)

    H = dq.tensor(w_a * dq.number(n_a), dq.eye(n_b)) \
      + dq.tensor(dq.eye(n_a), w_b_0 * dq.number(n_b)) \
      + H_drive + H_pump

    jumps = [jnp.sqrt(2*kappa) * dq.tensor(dq.eye(n_a), b)]

    save_extra = lambda rho: {"state_a" : dq.ptrace(rho, 0), "state_b" : dq.ptrace(rho, 1)}

    # run simulation (dynamiqs 0.3.6: options are keyword arguments)
    result = dq.mesolve(H, jumps, psi_0, tsave, save_extra=save_extra, save_states=False)

    state_a = result.extra["state_a"]
    state_b = result.extra["state_b"]
    w_eff = (w_d + w_p) / 2
    U_list = [dq.expm(1j * w_eff * dq.number(n_a) * t) for t in tsave]
    state_a_rotating = [U @ s @ U.dag() for U, s in zip(U_list, state_a)]

    # Same grid as dq.plot.wigner's defaults, which the original plotted with.
    xvec, yvec, W = dq.wigner(state_a_rotating[-1], xmax=5.0, npixels=101)

    def arr(x):
        return np.asarray(dq.to_jax(x))

    return {
        "label": label,
        "epsilon_0": float(epsilon_0),
        "ratio": float(ratio),
        "g2": float(g2),
        "epsilon_d": float(epsilon_d),
        "kappa_2": float(kappa_2),
        "alpha_init": complex(alpha),
        "alpha_circle": float(np.sqrt(epsilon_d / (2 * g2))),
        "n_a": n_a,
        "n_b": n_b,
        "tsave": np.asarray(tsave),
        "state_a": arr(state_a),
        "state_b": arr(state_b),
        "state_a_rotating": np.stack([arr(s) for s in state_a_rotating]),
        "wigner_end": (np.asarray(xvec), np.asarray(yvec), np.asarray(W)),
        "norm_elmul_end": float(dq.norm(state_a[-1].elmul(state_a[-1]))),
        "entropy_vn_end": float(dq.entropy_vn(state_a[-1])),
        "device": str(jax.devices()[0]),
        "elapsed_s": time.time() - t_start,
    }


def check_truncation(*, label, epsilon_0, n_a, T, n_time, ratio=10.0, precision="single"):
    """`run_single` of the original: max_t |1 - <[a, a^dag]>| and the same for b."""
    import time

    import jax.numpy as jnp
    import numpy as np

    import dynamiqs as dq

    t_start = time.time()
    dq.set_precision(precision)

    # Static pump (w_p = 2 w_a - w_d = 0): sin(epsilon_0), not J_1(epsilon_0).
    g2 = np.sin(epsilon_0) * E_J * phi_a**2 * phi_b
    epsilon_d = ratio * g2
    tsave = jnp.linspace(0, T, n_time)

    n_b = (n_a + 1) // 2
    a = dq.destroy(n_a)
    b = dq.destroy(n_b)

    varphi_a_op_tot = dq.tensor(jnp.sqrt(Z_a/2)*(a + a.dag()) / phi_quantum, dq.eye(n_b))
    varphi_b_op_tot = dq.tensor(dq.eye(n_a), jnp.sqrt(Z_b/2)*(b + b.dag()) / phi_quantum)

    psi_0 = dq.tensor(dq.fock(n_a, 1), dq.fock(n_b, 0))

    w_d = w_b_0
    drive = lambda t: epsilon_d * jnp.cos(w_d * t)
    H_drive = dq.modulated(drive, dq.tensor(dq.eye(n_a), b + b.dag()))

    w_p=2*w_a-w_d
    pump = lambda t: - 2 * E_J * jnp.sin(epsilon_0 * jnp.cos(w_p * t))
    H_pump = dq.modulated(pump, dq.sinm(varphi_a_op_tot + varphi_b_op_tot) - varphi_a_op_tot - varphi_b_op_tot)

    H = dq.tensor(w_a * dq.number(n_a), dq.eye(n_b)) \
      + dq.tensor(dq.eye(n_a), w_b_0 * dq.number(n_b)) \
      + H_drive + H_pump

    jumps = [jnp.sqrt(2*kappa) * dq.tensor(dq.eye(n_a), b)]
    result = dq.mesolve(H, jumps, psi_0, tsave)

    aadag_mean = dq.expect(dq.tensor(a@a.dag(), dq.eye(n_b)), result.states)
    adaga_mean = dq.expect(dq.tensor(a.dag()@a, dq.eye(n_b)), result.states)
    bbdag_mean = dq.expect(dq.tensor(dq.eye(n_a), b@b.dag()), result.states)
    bdagb_mean = dq.expect(dq.tensor(dq.eye(n_a), b.dag()@b), result.states)

    delta_a = jnp.max(jnp.abs(1 - (aadag_mean - adaga_mean).real))
    delta_b = jnp.max(jnp.abs(1 - (bbdag_mean - bdagb_mean).real))

    return {"label": label, "n_a": n_a, "n_b": n_b, "epsilon_0": float(epsilon_0),
            "delta_a": float(delta_a), "delta_b": float(delta_b),
            "elapsed_s": time.time() - t_start}

### The coordinator

One job per section, one GPU task per point. The coordinator only fans out and
gathers, so it is submitted with `entrypoint_num_cpus=0`, which keeps it on the
head node rather than on a freshly started spot worker (see
`/workspace/ray_driver_placement.md`). If that ever misbehaves, drop the
argument: the job then runs exactly as `acr.run`'s default.

Each point is caught on its own: a task lost to the infrastructure (node
preempted, OOM) comes back as an `"error"` entry instead of sinking the rest.

In [ ]:
import time

# Per GPU task. One whole GPU each: jax preallocates most of the card's memory.
TASK_RESOURCES = dict(num_gpus=1, num_cpus=2)


def sweep(worker, points, common, resources):
    """Coordinator: fans one GPU task out per point, gathers as they land.

    `worker` is passed in because it is defined in this notebook and ships by
    value. Results come back in the order of `points`.
    """
    import ray

    from anb_compute import ray as acr

    refs = {acr.submit(worker, **resources, **common, **p): i
            for i, p in enumerate(points)}
    out = [None] * len(points)
    pending = list(refs)
    while pending:
        ready, pending = ray.wait(pending, num_returns=1)
        i = refs[ready[0]]
        try:
            res = ray.get(ready[0])
        except Exception as exc:  # a task lost to infra, not to numerics
            res = {**points[i], "error": f"{type(exc).__name__}: {exc}"}
        out[i] = res
        print(f"{res['label']:<28} "
              + (res["error"] if "error" in res else f"{res['elapsed_s']:.1f} s"))
    return out


def run_on_cluster(worker, points, common):
    t0 = time.time()
    results = acr.run(
        sweep, worker, points, common, TASK_RESOURCES,
        poll_seconds=5,
        entrypoint_num_cpus=0,  # coordinator on the head, never on a spot node
    )
    print(f"\nwall clock, incl. job startup: {time.time() - t0:.1f} s "
          f"for {len(results)} point(s)")
    for r in results:
        if "error" in r:
            print(f"FAILED {r['label']}: {r['error']}")
    return results

In [ ]:
import matplotlib.pyplot as plt

THETA = np.linspace(0, 2 * np.pi, 100)


def plot_wigner(res, ax=None, title=None):
    """The worker's Wigner grid of the final memory state (rotating frame), with
    the circle |alpha| = sqrt(epsilon_d / (2 g2)), as `dq.plot.wigner` drew it."""
    if ax is None:
        _, ax = plt.subplots(figsize=(5.0, 4.4), constrained_layout=True)
    xvec, yvec, W = res["wigner_end"]
    vmax = 2 / np.pi
    # W is indexed [x, y]: transpose for pcolormesh, clip to +-2/pi.
    im = ax.pcolormesh(xvec, yvec, np.clip(W.T, -vmax, vmax), cmap="RdBu_r",
                       vmin=-vmax, vmax=vmax, shading="auto")
    r = res["alpha_circle"]
    ax.plot(r * np.cos(THETA), r * np.sin(THETA), color="r", linewidth=1, linestyle="--")
    ax.set_aspect("equal")
    ax.set_xlabel(r"$\mathrm{Re}(\alpha)$")
    ax.set_ylabel(r"$\mathrm{Im}(\alpha)$")
    ax.set_title(title or res["label"], fontsize=9)
    ax.figure.colorbar(im, ax=ax, ticks=[-vmax, 0, vmax], label="W")
    return ax

# Chaotic and stabilization regimes

The two regimes of the original, `epsilon_0 = pi/2` (chaotic) and
`epsilon_0 = 0.12` (stabilized), at `T = 5 / kappa`, in one job.

In [ ]:
# time discretization: reduce points if possible
T = 5 / kappa
n_time = 2   # 10000 -> 2000 for speed; increase if needed
n_a = 20

REGIMES = [
    dict(label="chaotic, eps_0 = pi/2", epsilon_0=0.5 * np.pi),
    dict(label="stabilized, eps_0 = 0.12", epsilon_0=0.12),
]
COMMON = dict(n_a=n_a, T=T, n_time=n_time, ratio=10.0, precision=PRECISION)

regimes = run_on_cluster(run_cat, REGIMES, COMMON)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.0, 4.4), constrained_layout=True)
for ax, res in zip(axes, regimes):
    if "error" in res:
        ax.set_title(f"{res['label']}: FAILED", fontsize=9)
        continue
    plot_wigner(res, ax=ax)
plt.show()

for res in regimes:
    if "error" not in res:
        print(f"{res['label']:<28} norm(rho_a * rho_a) = {res['norm_elmul_end']:.6g}   "
              f"S_vn(rho_a) = {res['entropy_vn_end']:.6g}   ({res['device']})")

# Sweep over the pump amplitude

`epsilon_p` over `linspace(0.05, 0.5, 20) * pi`, at `T = 10 / kappa`, one GPU
task per point.

The original loop set `alpha = 1j * epsilon_d / (4 g2)` before drawing the
circle, so the circle's real part was zero and it did not show. It is drawn
here with `sqrt(epsilon_d / (2 g2))`, as in the two regime plots above.

In [ ]:
epsilon_p_list = np.linspace(0.05, 0.5, 20) * np.pi

# time discretization: reduce points if possible
T = 10 / kappa
n_time = 2   # 10000 -> 2000 for speed; increase if needed
n_a = 20

POINTS = [dict(label=f"eps_p = {eps / np.pi:.3f} pi", epsilon_0=float(eps))
          for eps in epsilon_p_list]
COMMON = dict(n_a=n_a, T=T, n_time=n_time, ratio=10.0, precision=PRECISION)

sweep_results = run_on_cluster(run_cat, POINTS, COMMON)

In [ ]:
ncols = 5
nrows = -(-len(sweep_results) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3.0 * nrows),
                         constrained_layout=True)
for ax, res in zip(axes.flat, sweep_results):
    if "error" in res:
        ax.set_title(f"{res['label']}: FAILED", fontsize=8)
        ax.axis("off")
        continue
    xvec, yvec, W = res["wigner_end"]
    vmax = 2 / np.pi
    ax.pcolormesh(xvec, yvec, np.clip(W.T, -vmax, vmax), cmap="RdBu_r",
                  vmin=-vmax, vmax=vmax, shading="auto")
    r = res["alpha_circle"]
    ax.plot(r * np.cos(THETA), r * np.sin(THETA), color="r", linewidth=1, linestyle="--")
    ax.set_aspect("equal")
    ax.set_title(res["label"], fontsize=8)
    ax.tick_params(labelsize=7)
for ax in list(axes.flat)[len(sweep_results):]:
    ax.axis("off")
plt.show()

# Optional: truncation check

The original `run_single`, never called there. `delta_a` and `delta_b` should
stay small: they measure how far `<[a, a^dag]>` drifts from 1 because of the
Fock cutoff. Off by default; one GPU task per `n_a`.

In [ ]:
RUN_TRUNCATION_CHECK = False

if RUN_TRUNCATION_CHECK:
    POINTS = [dict(label=f"n_a = {n}", n_a=n) for n in (16, 20, 24)]
    COMMON = dict(epsilon_0=0.5 * np.pi, T=5 / kappa, n_time=2, ratio=10.0,
                  precision=PRECISION)
    for res in run_on_cluster(check_truncation, POINTS, COMMON):
        if "error" not in res:
            print(f"{res['label']:<10} n_b = {res['n_b']:<3} "
                  f"delta_a = {res['delta_a']:.3e}   delta_b = {res['delta_b']:.3e}")